# NusaSynth Post-Run Check (Architecture Validation)

Notebook ini fokus ke **arsitektur & threshold validation** — bukan untuk laporan thesis Bab IV.

## Tujuan:
- Validate apakah threshold/component pilihan sudah tepat
- Detect potential issues yang perlu adjustment di pipeline
- Inform decisions: tune threshold? per-lang prompt? re-train classifier?

## Yang dicek:
1. **Sanity Check** — pipeline integrity (CSV vs JSONL count match)
2. **Jaccard Threshold Validation** — apakah `DEDUP_THRESHOLD=0.5` tepat?
3. **Test Leakage Check** — pastikan synth tidak bocor ke test (data integrity)
4. **Architecture Tuning Signal** — per-lang reject pattern → suggest tuning
5. **PSLM Length Filter Validation** — apakah length filter berfungsi (std_ratio synth/seed)
6. **Retry Over-correction Detection** — chains >=3 attempts (signal: prompt issue)
7. **Manual Spot Check** — visual inspection random samples

> **Untuk laporan thesis Bab IV** (filter rates, BLEU, vocab novelty, dll): lihat `bab_iv_metrics.ipynb`.

## Setup

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_colwidth', 150)
pd.set_option('display.width', 200)

ROOT = Path('y:/Michh/Python/Projects/MAGenerator')
LANGS = ['ace', 'ban', 'bjn', 'jav', 'mad', 'min', 'sun']
SENTENCES_PER_SEED = 5

# Load: seed, synthetic, valid, test, log
seed_dfs, syn_dfs, valid_dfs, test_dfs, log_dfs = {}, {}, {}, {}, {}
for lang in LANGS:
    seed_p  = ROOT / f'data/nusax_senti/{lang}/train.csv'
    syn_p   = ROOT / f'outputs/synthetic/{lang}/synthetic.csv'
    valid_p = ROOT / f'data/nusax_senti/{lang}/valid.csv'
    test_p  = ROOT / f'data/nusax_senti/{lang}/test.csv'
    log_p   = ROOT / f'outputs/synthetic/{lang}/pipeline_log.jsonl'
    if not all(p.exists() for p in [seed_p, syn_p, log_p, valid_p, test_p]):
        continue
    seed_dfs[lang]  = pd.read_csv(seed_p)
    syn_dfs[lang]   = pd.read_csv(syn_p)
    valid_dfs[lang] = pd.read_csv(valid_p)
    test_dfs[lang]  = pd.read_csv(test_p)
    with open(log_p, encoding='utf-8') as f:
        log_dfs[lang] = pd.DataFrame([json.loads(line) for line in f])
    print(f'  [{lang}] OK')


def jaccard_bigram(s1, s2):
    w1, w2 = s1.lower().split(), s2.lower().split()
    bg1, bg2 = set(zip(w1, w1[1:])), set(zip(w2, w2[1:]))
    if not bg1 or not bg2:
        return 0.0
    return len(bg1 & bg2) / len(bg1 | bg2)

  [ace] OK
  [ban] OK
  [bjn] OK
  [jav] OK
  [mad] OK
  [min] OK
  [sun] OK


## 1. Sanity Check (Pipeline Integrity)

Verify CSV row count == JSONL accepted count. Mismatch → ada bug di save_results.

In [3]:
rows = []
for lang in LANGS:
    if lang not in log_dfs:
        continue
    log_df, syn_df, seed_df = log_dfs[lang], syn_dfs[lang], seed_dfs[lang]
    o = log_df['outcome'].value_counts()
    n_acc = o.get('accepted', 0)
    expected = len(seed_df) * SENTENCES_PER_SEED
    rows.append({
        'lang': lang,
        'csv_rows': len(syn_df),
        'accepted': n_acc,
        'match': 'OK' if len(syn_df) == n_acc else 'MISMATCH',
        'expected': expected,
        'yield%': round(n_acc / expected * 100, 2),
        'discarded': o.get('discarded', 0),
        'dedup_filtered': o.get('dedup_filtered', 0),
        'retried_events': o.get('retried', 0),
        'batch_failed': o.get('batch_failed', 0),
    })
display(pd.DataFrame(rows))

,lang,csv_rows,accepted,match,expected,yield%,discarded,dedup_filtered,retried_events,batch_failed
0,ace,2444,2444,OK,2500,97.76,1,55,200,0
1,ban,2465,2465,OK,2500,98.60,2,33,180,0
2,bjn,2473,2473,OK,2500,98.92,5,22,202,0
3,jav,2498,2498,OK,2500,99.92,1,1,144,0
4,mad,2448,2448,OK,2500,97.92,3,49,253,0
5,min,2479,2479,OK,2500,99.16,2,19,133,0
6,sun,2479,2479,OK,2500,99.16,3,18,152,0


## 2. Jaccard Threshold Validation

**Pertanyaan**: apakah `DEDUP_THRESHOLD=0.5` (di [config.py](../NusaSynth/config.py)) appropriate?

**Metodologi**: untuk tiap kalimat di accepted set (sample 500), hitung max Jaccard ke tetangga terdekat. Lihat distribusi p50/p90/p99 dan counter-factual filter count di threshold 0.3-0.7.

**Cara baca**:
- `above_0.5` ~0 → threshold jarang hit, design conservative (bisa relax atau biarkan sebagai safety net)
- `above_0.4 - above_0.5` besar → ada banyak near-dup di [0.4, 0.5) yang lolos → tighten ke 0.4
- `p99 < 0.3` → sangat diverse, threshold mana saja juga aman

In [4]:
JACCARD_SAMPLE_SIZE = 500

rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    texts = syn_dfs[lang]['text'].tolist()
    if len(texts) > JACCARD_SAMPLE_SIZE:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(texts), JACCARD_SAMPLE_SIZE, replace=False)
        texts = [texts[i] for i in idx]
    n = len(texts)
    max_jaccards = []
    for i in range(n):
        max_j = 0.0
        for j in range(n):
            if i == j:
                continue
            jac = jaccard_bigram(texts[i], texts[j])
            if jac > max_j:
                max_j = jac
        max_jaccards.append(max_j)
    rows.append({
        'lang': lang,
        'p50': round(float(np.percentile(max_jaccards, 50)), 3),
        'p90': round(float(np.percentile(max_jaccards, 90)), 3),
        'p99': round(float(np.percentile(max_jaccards, 99)), 3),
        'max': round(float(np.max(max_jaccards)), 3),
        'above_0.3': sum(1 for m in max_jaccards if m >= 0.3),
        'above_0.4': sum(1 for m in max_jaccards if m >= 0.4),
        'above_0.5': sum(1 for m in max_jaccards if m >= 0.5),
        'above_0.6': sum(1 for m in max_jaccards if m >= 0.6),
        'above_0.7': sum(1 for m in max_jaccards if m >= 0.7),
    })

display(pd.DataFrame(rows))

,lang,p50,p90,p99,max,above_0.3,above_0.4,above_0.5,above_0.6,above_0.7
0,ace,0.054,0.238,0.465,0.483,23,16,0,0,0
1,ban,0.040,0.176,0.333,0.440,12,2,0,0,0
2,bjn,0.052,0.148,0.385,0.455,14,4,0,0,0
3,jav,0.042,0.093,0.273,0.447,2,2,0,0,0
4,mad,0.045,0.218,0.462,0.474,27,14,0,0,0
5,min,0.048,0.162,0.419,0.455,12,6,0,0,0
6,sun,0.047,0.137,0.333,0.429,14,2,0,0,0


## 3. Test Leakage Check

**Risiko**: kalau synth terlalu mirip ke test → downstream eval inflated.

**Metodologi**: sample 200 synth per bahasa, hitung max Jaccard ke seluruh test.

**Acceptable**:
- p99 max-Jaccard < 0.4 → no leakage
- `leak_>=0.5` ~0 → aman
- `leak_>=0.7` HARUS 0 → kalau ada, **wajib** filter sebelum training

In [5]:
LEAK_SAMPLE = 200

rows = []
for lang in LANGS:
    if lang not in syn_dfs or lang not in test_dfs:
        continue
    syn_texts = syn_dfs[lang]['text'].tolist()
    test_texts = test_dfs[lang]['text'].tolist()
    if len(syn_texts) > LEAK_SAMPLE:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(syn_texts), LEAK_SAMPLE, replace=False)
        syn_texts = [syn_texts[i] for i in idx]
    max_jacs = []
    leak_03 = leak_05 = leak_07 = 0
    for s in syn_texts:
        max_j = 0.0
        for tt in test_texts:
            j = jaccard_bigram(s, tt)
            if j > max_j:
                max_j = j
        max_jacs.append(max_j)
        if max_j >= 0.3: leak_03 += 1
        if max_j >= 0.5: leak_05 += 1
        if max_j >= 0.7: leak_07 += 1
    rows.append({
        'lang': lang,
        'p50': round(float(np.percentile(max_jacs, 50)), 3),
        'p90': round(float(np.percentile(max_jacs, 90)), 3),
        'p99': round(float(np.percentile(max_jacs, 99)), 3),
        'max': round(float(np.max(max_jacs)), 3),
        'leak_>=0.3': leak_03,
        'leak_>=0.5': leak_05,
        'leak_>=0.7': leak_07,
    })

display(pd.DataFrame(rows))

,lang,p50,p90,p99,max,leak_>=0.3,leak_>=0.5,leak_>=0.7
0,ace,0.029,0.050,0.071,0.111,0,0,0
1,ban,0.023,0.040,0.077,0.095,0,0,0
2,bjn,0.033,0.058,0.082,0.083,0,0,0
3,jav,0.027,0.050,0.077,0.125,0,0,0
4,mad,0.026,0.048,0.118,0.125,0,0,0
5,min,0.032,0.050,0.073,0.100,0,0,0
6,sun,0.025,0.045,0.072,0.111,0,0,0


## 4. Architecture Tuning Signal (per-lang reject pattern)

Reject pattern berbeda antar bahasa → opportunity untuk tuning per-lang.

**Aturan suggestion**:
- LV% > 30% → strengthen language constraint di Generator prompt (atau add few-shot bahasa target)
- PSLM% > 50% → tighten target length / lower temperature
- SV% > 40% → **verify NusaBERT classifier reliability dulu** (jangan-jangan classifier yang lemah, bukan LLM)
- dedup_rate% > 2% → Contextualizer perlu more variation diversity

In [6]:
def classify_reject(row):
    sv = row.get('sv_verdict')
    lv = row.get('lv_verdict')
    if sv == 'VALIDATION_FAILED' or lv == 'VALIDATION_FAILED':
        return 'VALIDATION_FAILED'
    if sv == 'REJECT':
        return 'SV'
    if lv == 'REJECT':
        return 'LV'
    return 'PSLM'

rows = []
for lang in LANGS:
    if lang not in log_dfs:
        continue
    log_df = log_dfs[lang]
    o = log_df['outcome'].value_counts()
    n_dedup = o.get('dedup_filtered', 0)
    n_acc = o.get('accepted', 0)
    rejected = log_df[log_df['outcome'].isin(['retried', 'discarded'])].copy()
    rej_total = len(rejected)
    if rej_total == 0:
        continue
    rejected['reason'] = rejected.apply(classify_reject, axis=1)
    sv_pct = (rejected['reason'] == 'SV').sum() / rej_total * 100
    lv_pct = (rejected['reason'] == 'LV').sum() / rej_total * 100
    pslm_pct = (rejected['reason'] == 'PSLM').sum() / rej_total * 100

    suggestions = []
    if lv_pct > 30:
        suggestions.append('strengthen lang constraint in Generator prompt')
    if pslm_pct > 50:
        suggestions.append('tighten target length / lower temperature')
    if sv_pct > 40:
        suggestions.append('verify NusaBERT validation F1 first')
    dedup_rate = n_dedup / (n_acc + n_dedup) * 100 if (n_acc + n_dedup) else 0
    if dedup_rate > 2:
        suggestions.append('Contextualizer needs more variation diversity')

    rows.append({
        'lang': lang,
        'SV%': round(sv_pct, 1),
        'LV%': round(lv_pct, 1),
        'PSLM%': round(pslm_pct, 1),
        'dedup_rate%': round(dedup_rate, 2),
        'suggestion': '; '.join(suggestions) if suggestions else 'OK as-is',
    })

display(pd.DataFrame(rows))

,lang,SV%,LV%,PSLM%,dedup_rate%,suggestion
0,ace,29.4,41.3,29.4,2.20,strengthen lang constraint in Generator prompt; Contextualizer needs more variation diversity
1,ban,37.9,25.8,36.3,1.32,OK as-is
2,bjn,38.6,16.9,44.4,0.88,OK as-is
3,jav,28.3,5.5,66.2,0.04,tighten target length / lower temperature
4,mad,27.7,33.2,39.1,1.96,strengthen lang constraint in Generator prompt
5,min,43.7,16.3,40.0,0.76,verify NusaBERT validation F1 first
6,sun,31.0,26.5,42.6,0.72,OK as-is


## 5. PSLM Length Filter Validation

**Pertanyaan**: apakah PSLM filter (`|gen_len - seed_len| > 1.0 × seed_std` di [collect_node](../NusaSynth/nodes.py)) benar-benar membatasi length drift?

**Cara cek**: ratio std synth vs seed per (lang, label). Target 0.80-1.10.
- ratio < 0.70 → **length collapse**: synth terlalu seragam, PSLM over-filtering atau LLM mode-collapse
- ratio > 1.20 → PSLM tidak cukup ketat, atau threshold 1.0σ terlalu longgar
- mean diff > 5 kata → systematic bias (Generator verbose/terse)

In [7]:
rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    seed_df, syn_df = seed_dfs[lang], syn_dfs[lang]
    seed_df = seed_df.copy(); syn_df = syn_df.copy()
    seed_df['wc'] = seed_df['text'].str.split().str.len()
    syn_df['wc']  = syn_df['text'].str.split().str.len()
    for label in ['negative', 'neutral', 'positive']:
        s_seed = seed_df[seed_df['label'] == label]['wc']
        s_syn  = syn_df[syn_df['label'] == label]['wc']
        if len(s_seed) > 1 and len(s_syn) > 1:
            ratio = round(s_syn.std() / s_seed.std(), 2)
            mean_diff = round(s_syn.mean() - s_seed.mean(), 1)
            verdict = 'OK' if 0.80 <= ratio <= 1.20 and abs(mean_diff) <= 5 else 'CHECK'
            rows.append({
                'lang': lang,
                'label': label,
                'seed_mean': round(s_seed.mean(), 1),
                'syn_mean': round(s_syn.mean(), 1),
                'mean_diff': mean_diff,
                'seed_std': round(s_seed.std(), 1),
                'syn_std': round(s_syn.std(), 1),
                'std_ratio': ratio,
                'verdict': verdict,
            })

display(pd.DataFrame(rows))

,lang,label,seed_mean,syn_mean,mean_diff,seed_std,syn_std,std_ratio,verdict
0,ace,negative,22.5,23.8,1.4,15.0,14.3,0.95,OK
1,ace,neutral,14.5,16.1,1.6,9.6,9.7,1.02,OK
2,ace,positive,31.2,32.2,1.0,15.4,14.3,0.93,OK
3,ban,negative,21.9,23.0,1.1,13.8,12.6,0.91,OK
4,ban,neutral,13.8,15.5,1.7,8.5,8.4,0.99,OK
5,ban,positive,30.8,30.7,-0.1,14.4,12.8,0.89,OK
6,bjn,negative,21.9,23.0,1.1,14.1,13.1,0.93,OK
7,bjn,neutral,13.5,14.7,1.2,8.3,8.2,0.98,OK
8,bjn,positive,30.6,30.8,0.2,14.4,13.0,0.90,OK
9,jav,negative,21.8,23.2,1.4,14.2,13.4,0.94,OK


## 5b. Length Gap — Synthetic vs TEST (bukan seed)

Cell 5 mengecek synthetic vs **SEED** (target PSLM). Cell ini mengecek synthetic vs **TEST**, relevan untuk downstream karena model dievaluasi di test.

**Temuan**: synthetic neutral MATCH seed (std_ratio ~1.0, cell 5) tapi TEST neutral punya tail lebih panjang (std ~11 vs ~8.5; p95 ~40 vs ~30; >30 kata ~9% vs ~5%). Ini **train/test distribution shift intrinsik NusaX**, BUKAN defect generator, pipeline meniru seed dengan setia dan seed sendiri yang tail-nya lebih tipis dari test.

In [ ]:
# 5b. Per (lang, label): bandingkan panjang seed vs synthetic vs TEST
def _ls(s):
    w = s.str.split().str.len()
    return round(w.mean(), 1), round(w.std(), 1), int(w.quantile(.95)), round((w > 30).mean() * 100, 1)

rows = []
for lang in LANGS:
    if lang not in syn_dfs:
        continue
    for label in ['negative', 'neutral', 'positive']:
        sm, ss, sp, s30 = _ls(seed_dfs[lang].query("label==@label")['text'])
        ym, ys, yp, y30 = _ls(syn_dfs[lang].query("label==@label")['text'])
        tm, ts, tp, t30 = _ls(test_dfs[lang].query("label==@label")['text'])
        rows.append({
            'lang': lang, 'label': label,
            'seed_std': ss, 'syn_std': ys, 'test_std': ts,
            'seed_p95': sp, 'syn_p95': yp, 'test_p95': tp,
            'seed_>30%': s30, 'syn_>30%': y30, 'test_>30%': t30,
        })
df = pd.DataFrame(rows)
print("synthetic MATCH seed (std/p95/>30% mirip) tapi TEST punya tail lebih panjang (terutama neutral).")
print("=> gap = train/test shift intrinsik, BUKAN defect generator. PSLM + prompt bekerja sesuai desain.")
print("\nNEUTRAL saja:")
display(df[df['label'] == 'neutral'])
print("Semua label:")
display(df)

## 5c. Downstream Error Mode — Panjang Kalimat Test yang Salah-Klasifikasi

> Butuh prediksi test dari model downstream (`tmp_pdf/test_predictions_aug.pkl` = NusaBERT-large AUGMENTED). Ini analisis **error model**, BUKAN dari pipeline synthetic, makanya tidak ada di `bab_iv_metrics.ipynb` (yang hanya punya teks, bukan prediksi).

**Temuan**: test neutral yang SALAH diklasifikasi jauh lebih panjang dari yang benar (≈24.8 vs 13.7 kata, Mann-Whitney p≈7.6e-15). Mengonfirmasi tail-panjang-neutral = titik lemah model.

⚠️ **Caveat (jangan over-claim)**: korelasi length→ΔF1 secara KAUSAL lemah (workflow H1 refuted, r≈−0.19..−0.34). Ini menjelaskan MODE error, tapi belum tentu mengisi tail neutral menaikkan F1. Synthetic sudah faithful ke seed (cell 5b) → gap tail = limitasi intrinsik dataset, bukan yang bisa diperbaiki augmentasi faithful.

In [ ]:
# 5c. Panjang test neutral: SALAH vs BENAR klasifikasi (butuh pickle prediksi downstream)
import pickle
from scipy.stats import mannwhitneyu

PKL = ROOT / 'tmp_pdf/test_predictions_aug.pkl'
if not PKL.exists():
    print(f'SKIP: {PKL} tidak ada (jalankan downstream eval dulu / sesuaikan path).')
else:
    LAB = ['negative', 'neutral', 'positive']
    norm = lambda xs: [LAB[i] if isinstance(i, (int, np.integer)) else i for i in xs]
    d = pickle.load(open(PKL, 'rb'))
    rows = []; allw = []; allr = []
    for lang in LANGS:
        if lang not in d:
            continue
        v = d[lang]; pr = norm(v['pred']); tr = norm(v['true']); tx = v['texts']
        w = [len(str(x).split()) for p, t, x in zip(pr, tr, tx) if t == 'neutral' and p != 'neutral']
        r = [len(str(x).split()) for p, t, x in zip(pr, tr, tx) if t == 'neutral' and p == 'neutral']
        allw += w; allr += r
        rows.append({'lang': lang, 'wrong_n': len(w), 'wrong_mean_w': round(np.mean(w), 1) if w else 0,
                     'right_n': len(r), 'right_mean_w': round(np.mean(r), 1) if r else 0})
    u, p = mannwhitneyu(allw, allr, alternative='two-sided')
    print('TEST neutral, dipisah benar/salah klasifikasi (model NusaBERT-large AUGMENTED):')
    print(f'ALL: salah n={len(allw)} mean={np.mean(allw):.1f}w | benar n={len(allr)} mean={np.mean(allr):.1f}w | Mann-Whitney p={p:.2e}')
    print(f'>30 kata: salah {(np.array(allw)>30).mean()*100:.1f}% vs benar {(np.array(allr)>30).mean()*100:.1f}%')
    display(pd.DataFrame(rows))

## 6. Retry Over-correction Detection

Chains dengan >=3 attempts berarti retry mengalami over-correction (fix A → broke B → discard).

**Healthy signal**:
- over_correction% < 1% → retry feedback efektif
- over_correction% > 5% → prompt feedback tidak guide LLM dengan baik, perlu revisi

In [8]:
chain_rows = []
sample_chains = {}

for lang in LANGS:
    if lang not in log_dfs:
        continue
    log_df = log_dfs[lang]
    if 'seed_id' not in log_df.columns or 'plan_id' not in log_df.columns:
        continue
    chains = log_df.dropna(subset=['plan_id', 'seed_id']).groupby(['seed_id', 'plan_id'])
    chain_lens = chains.size()
    dist = chain_lens.value_counts().sort_index().to_dict()
    total = sum(dist.values())
    chain_rows.append({
        'lang': lang,
        '1_attempt': dist.get(1, 0),
        '2_attempts': dist.get(2, 0),
        '3_attempts': dist.get(3, 0),
        'over_correction%': round(dist.get(3, 0) / total * 100, 2) if total else 0,
    })
    long_chains = chain_lens[chain_lens >= 3].index.tolist()
    if long_chains:
        sample_chains[lang] = long_chains[0]

display(pd.DataFrame(chain_rows))

print('\nOver-correction samples:')
for lang, (sid_seed, pid) in sample_chains.items():
    log_df = log_dfs[lang]
    chain = log_df[(log_df['seed_id'] == sid_seed) & (log_df['plan_id'] == pid)].sort_values('retry_count')
    cols = [c for c in ['retry_count', 'sid', 'outcome', 'sv_verdict', 'lv_verdict', 'prev_feedback'] if c in chain.columns]
    print(f'\n[{lang}] seed={sid_seed} plan={pid}')
    display(chain[cols])

,lang,1_attempt,2_attempts,3_attempts,over_correction%
0,ace,2313,174,13,0.52
1,ban,2323,174,3,0.12
2,bjn,2311,176,13,0.52
3,jav,2358,140,2,0.08
4,mad,2258,231,11,0.44
5,min,2371,125,4,0.16
6,sun,2355,138,7,0.28



Over-correction samples:

[ace] seed=6 plan=3


,retry_count,sid,outcome,sv_verdict,lv_verdict,prev_feedback
1309,0,3,retried,REJECT,NaN,"The sentence expresses disappointment that the reality of the location did not live up to the viral videos, which introduces a negative sentiment."
1331,1,25,retried,PASS,PASS,Length drift: generated sentence (82 words) deviates too far from seed length (69 words). Generate a sentence with elaboration depth closer to the...
1334,2,28,accepted,PASS,PASS,Length drift: generated sentence (82 words) deviates too far from seed length (69 words). Generate a sentence with elaboration depth closer to the...



[ban] seed=311 plan=14


,retry_count,sid,outcome,sv_verdict,lv_verdict,prev_feedback
1289,0,14,retried,REJECT,NaN,The sentence is a complaint about deceptive packaging and poor value for money.
1308,1,33,retried,REJECT,NaN,The sentence expresses a negative sentiment by complaining about misleading packaging and insufficient food portions.
1309,2,34,discarded,REJECT,NaN,The sentence expresses a negative sentiment by complaining about misleading packaging and insufficient food portions.


...

,retry_count,sid,outcome,sv_verdict,lv_verdict,prev_feedback
1320,0,0,retried,PASS,REJECT,The sentence contains a logical contradiction (being alone vs being with a wife) and uses the non-existent/incoherent word 'kerrong-toronana' inst...
1345,1,25,retried,PASS,PASS,Length drift: generated sentence (73 words) deviates too far from seed length (62 words). Generate a sentence with elaboration depth closer to the...
1347,2,27,accepted,PASS,PASS,Length drift: generated sentence (73 words) deviates too far from seed length (62 words). Generate a sentence with elaboration depth closer to the...



[min] seed=43 plan=14


,retry_count,sid,outcome,sv_verdict,lv_verdict,prev_feedback
1549,0,14,retried,REJECT,NaN,The word 'sajuak' (cool/refreshing) conveys positive sentiment.
1565,1,30,retried,PASS,PASS,Length drift: generated sentence (18 words) deviates too far from seed length (6 words). Generate a sentence with elaboration depth closer to the ...
1566,2,31,accepted,PASS,PASS,Length drift: generated sentence (18 words) deviates too far from seed length (6 words). Generate a sentence with elaboration depth closer to the ...



[sun] seed=63 plan=21


,retry_count,sid,outcome,sv_verdict,lv_verdict,prev_feedback
1431,0,21,retried,REJECT,NaN,The sentence expresses a negative situation regarding unexpected credit loss and premium charges.
1437,1,27,retried,REJECT,NaN,"The sentence describes a negative event (automatic credit deduction), which carries a negative sentiment rather than a neutral one."
1439,2,29,accepted,PASS,PASS,"The sentence describes a negative event (automatic credit deduction), which carries a negative sentiment rather than a neutral one."


## 7. Manual Spot Check

Random samples per label per bahasa untuk visual inspection:
- Bahasa pure (tidak ada code-mixing visible)?
- Sentimen match label?
- Variety topic/style (bukan template repetitive)?

In [9]:
SPOT_LANGS = LANGS  # ubah ke ['jav'] dst untuk subset
N_SAMPLES = 5

for lang in SPOT_LANGS:
    if lang not in syn_dfs:
        continue
    syn_df = syn_dfs[lang]
    print(f'\n=== {lang} (n={len(syn_df)}) ===')
    for label in ['negative', 'neutral', 'positive']:
        sub = syn_df[syn_df['label'] == label]
        if len(sub) == 0:
            continue
        print(f'\n[{label}] n={len(sub)}')
        sample = sub.sample(min(N_SAMPLES, len(sub)), random_state=42)
        for _, row in sample.iterrows():
            print(f'[{row["id"]}] {row["text"]}')


=== ace (n=2444) ===

[negative] n=925
[4] Buku nyoe geupeugah haba hana ilop lam akai! Cara pasang kipas nyoe jipeugah meunoe-meudeh tapi hana tateupeu sapeu pih. Teori dari pane nyan?
[11] Teulah that. Lon peusan bu rames poh sa malam, ka lheueh lon bayeue pakek saldo. Driver hana teuka-teuka. Tiba-tiba bak aplikasi jipeugah pesanan ka lheueh jiteuka padahal lon mantong lon preeh. Meunyo hana jiteuka pakon jipeugah ka jikuat.
[20] Kolam nyoe that teulah tabloe tiket, ie jih kuto. Na karcis tamong per sidroe ureueng 25.000, keu sewa ban 10.000.
[15] Kerana peunasaran ngen foto-foto lagak bak internet, sengaja mita watee luweng jak u pantee nyan. Phon seumangat, peubla macet, niatlah pokok jih keumeung meureuwi. Tapi keucewe ngen jalan jih nyang brok that dan juoh, jadi jih meutamah beue bak badan. Suasana pantee nyan cit unik, tapi ureung hana that leut di sinan sang-sang. Pemandangan lumayan kreatif, man golom sebandeng ngen rasa hek bak jalan nyang hancoe.
[10] Tulak keudeh tulak k

## Verdict

**Architecture sehat kalau**:
- Sanity check: CSV == JSONL accepted (no MISMATCH)
- Jaccard p99 < 0.5 → threshold longgar (jarang hit), tapi tetap berfungsi sebagai safety net
- Test leakage `leak_>=0.7 == 0` dan `leak_>=0.5` ~0
- Architecture tuning: suggestion = 'OK as-is' atau hanya minor flag
- PSLM std_ratio 0.80-1.10 di semua (lang, label), |mean_diff| <= 5
- Over-correction% < 1%
- Spot check: bahasa pure, sentiment match, ada variasi

**Action items kalau ada flag**:
1. **Jaccard `above_0.5` banyak** → tighten threshold ke 0.4 di `config.py`
2. **`leak_>=0.5` > 0** → manual filter synth dengan max-Jaccard ke test >= 0.5 sebelum merge ke train
3. **LV% > 30%** → strengthen language constraint di [prompts.py](../NusaSynth/prompts.py) Generator section
4. **PSLM% > 50%** (tinggi reject) ATAU **std_ratio < 0.70** (length collapse) → tighten target length atau lower temperature
5. **SV% > 40%** → re-evaluate NusaBERT validation F1, jangan-jangan classifier-nya yang bermasalah
6. **Over-correction% > 5%** → revisi `prev_feedback` template di retry mechanism